# RecA FDA-Approved Drug Repurposing, SHAP, ADMET, and Bayesian Good/Bad Fingerprint Workflow

**Publication-ready notebook revised to follow the senior workflow style**

This notebook integrates the RecA FDA-drug repurposing pipeline with model interpretation, ADMET-likeness triage, and Bayesian Good/Bad Fingerprint profiling. The workflow is arranged as a clean, reproducible notebook rather than a pasted script.

Main workflow:

1. Load the RecA ChEMBL-derived feature-selection matrix.
2. Train and benchmark classical ML models.
3. Predict active-like probability for FDA-approved compounds.
4. Prioritize candidates using ML probability and ADMET-likeness rules.
5. Interpret model predictions using SHAP.
6. Identify Bayesian Good/Bad fingerprint bits from the RecA training set.
7. Apply Bayesian fingerprint scoring to FDA candidates.
8. Export publication-ready tables and figures.

> **Publication note:** The Bayesian Good/Bad Fingerprint model is fitted using the training split for unbiased hold-out evaluation, then a final full-data Bayesian profile is generated for candidate prioritization.


## 0. Notebook notes

Before running this notebook, the earlier notebooks should already have generated the RecA feature-selection and FDA fingerprint outputs. The expected default files are:

- `outputs/feature_selection/03_recA_low_variance_filtered.csv`
- `outputs/feature_selection/03_recA_fscore_ranking.csv`
- `outputs/fda_prediction/05_fda_combined_fingerprints.csv`
- `outputs/fda_prediction/05_fda_recA_predictions.csv` *(optional; generated again if absent)*

The notebook is designed to be reproducible in **Google Colab**, **Jupyter Notebook**, or a local Python environment. If your folder names are different, update the configuration cell only.


In [ ]:
# ============================================================
# Optional installation cell
# Run only if the packages are not already installed.
# ============================================================

# !pip install -q pandas numpy matplotlib scikit-learn shap xgboost catboost


In [ ]:
# ============================================================
# 1. Import libraries
# ============================================================

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Iterable, Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    import shap
except ImportError:
    shap = None

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from catboost import CatBoostClassifier
except ImportError:
    CatBoostClassifier = None

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded successfully.')


In [ ]:
# ============================================================
# 2. Project configuration
# ============================================================

PROJECT_DIR = Path.cwd()

# Main inputs from previous RecA workflows
TRAINING_DATA_CANDIDATES = [
    PROJECT_DIR / 'outputs' / 'feature_selection' / '03_recA_low_variance_filtered.csv',
    PROJECT_DIR / 'outputs' / 'recA_chembl' / 'feature_selection' / 'recA_low_variance_filtered.csv',
    PROJECT_DIR / '03_recA_low_variance_filtered.csv',
]

F_SCORE_CANDIDATES = [
    PROJECT_DIR / 'outputs' / 'feature_selection' / '03_recA_fscore_ranking.csv',
    PROJECT_DIR / 'outputs' / 'recA_chembl' / 'feature_selection' / 'recA_fscore_ranking.csv',
    PROJECT_DIR / '03_recA_fscore_ranking.csv',
]

FDA_FINGERPRINT_CANDIDATES = [
    PROJECT_DIR / 'outputs' / 'fda_prediction' / '05_fda_combined_fingerprints.csv',
    PROJECT_DIR / 'outputs' / 'recA_chembl' / 'fda_prediction' / 'fda_combined_fingerprints.csv',
    PROJECT_DIR / '05_fda_combined_fingerprints.csv',
]

FDA_PREDICTION_CANDIDATES = [
    PROJECT_DIR / 'outputs' / 'fda_prediction' / '05_fda_recA_predictions.csv',
    PROJECT_DIR / 'outputs' / 'fda_prediction' / '05_fda_predictions_with_docking.csv',
    PROJECT_DIR / 'outputs' / 'recA_chembl' / 'docking_real' / 'fda_predictions_with_real_docking.csv',
    PROJECT_DIR / '05_fda_recA_predictions.csv',
]

OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'fda_shap_admet_publication'
FIGURE_DIR = OUTPUT_DIR / 'figures'
TABLE_DIR = OUTPUT_DIR / 'tables'
MODEL_DIR = OUTPUT_DIR / 'models'

for folder in [OUTPUT_DIR, FIGURE_DIR, TABLE_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

TOP_K_FEATURES = 100
TEST_SIZE = 0.20
ACTIVE_PROBABILITY_THRESHOLD = 0.50
TOP_N_CANDIDATES = 20
TOP_N_SHAP_LOCAL = 5

print('Project directory:', PROJECT_DIR)
print('Output directory:', OUTPUT_DIR)


## 3. Utility functions

These helper functions keep the notebook robust when file names differ slightly between your workflow and the senior workflow.


In [ ]:
# ============================================================
# 3. Utility functions
# ============================================================

def first_existing_path(candidates: Iterable[Path], label: str, required: bool = True) -> Optional[Path]:
    """Return the first existing file from a list of candidate paths."""
    for path in candidates:
        if path.exists():
            print(f'{label}: {path}')
            return path
    if required:
        candidate_text = "\n".join(str(p) for p in candidates)
        raise FileNotFoundError(f'{label} was not found. Checked:\n{candidate_text}')
    print(f'{label}: not found; this step will be skipped if optional.')
    return None


def detect_label_column(df: pd.DataFrame) -> str:
    """Detect the class/label column from common RecA workflow names."""
    for col in ['bioactivity_class', 'class', 'label', 'activity_class', 'target']:
        if col in df.columns:
            return col
    raise ValueError('No label column found. Expected one of: bioactivity_class, class, label, activity_class, target.')


def convert_label_to_binary(series: pd.Series) -> pd.Series:
    """Convert common active/inactive labels to binary values."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype(int)

    mapping = {
        'active': 1,
        'active_like': 1,
        '1': 1,
        'inactive': 0,
        'inactive_like': 0,
        '0': 0,
    }
    converted = series.astype(str).str.lower().str.strip().map(mapping)
    if converted.isna().any():
        missing_values = sorted(series[converted.isna()].astype(str).unique())
        raise ValueError(f'Unrecognized class labels: {missing_values}')
    return converted.astype(int)


def clean_feature_matrix(df: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    """Force selected feature columns to numeric values and replace missing values with zero."""
    X = df[feature_columns].copy()
    X = X.replace([np.inf, -np.inf], np.nan)
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    return X.fillna(0)


def save_current_figure(path: Path) -> None:
    """Save current Matplotlib figure with publication-quality settings."""
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches='tight')
    print('Figure saved to:', path)


## 4. Load RecA training matrix and selected features

The low-variance-filtered RecA matrix is used as the modeling input. The F-score ranking file is used to select the top molecular fingerprint descriptors.


In [ ]:
# ============================================================
# 4. Load RecA training data and selected features
# ============================================================

TRAINING_DATA_FILE = first_existing_path(TRAINING_DATA_CANDIDATES, 'Training data file')
F_SCORE_FILE = first_existing_path(F_SCORE_CANDIDATES, 'F-score ranking file')

training_df = pd.read_csv(TRAINING_DATA_FILE)
fscore_df = pd.read_csv(F_SCORE_FILE)

label_col = detect_label_column(training_df)
y = convert_label_to_binary(training_df[label_col])

# Detect feature names from F-score ranking file.
possible_feature_cols = ['feature', 'Feature', 'descriptor', 'Descriptor', 'fingerprint', 'Fingerprint']
feature_name_col = next((col for col in possible_feature_cols if col in fscore_df.columns), None)

if feature_name_col is None:
    # Fallback: first column is treated as feature name.
    feature_name_col = fscore_df.columns[0]

ranked_features = [f for f in fscore_df[feature_name_col].astype(str).tolist() if f in training_df.columns]
selected_features = ranked_features[:TOP_K_FEATURES]

if len(selected_features) == 0:
    metadata_cols = {'Name', 'canonical_smiles', 'bioactivity_class', 'class', 'label', 'activity_class', 'target'}
    selected_features = [col for col in training_df.columns if col not in metadata_cols]
    selected_features = selected_features[:TOP_K_FEATURES]

X = clean_feature_matrix(training_df, selected_features)

print('Training matrix shape:', training_df.shape)
print('Selected feature count:', len(selected_features))
print('Class distribution:')
print(y.value_counts().rename(index={0: 'inactive_like', 1: 'active_like'}))
X.head()


## 5. Train and benchmark publication-ready models

This section keeps the senior SVM workflow style but adds additional classical ML models for benchmarking. The best model is selected using ROC-AUC when available, followed by F1-score.


In [ ]:
# ============================================================
# 5. Train/test split and model benchmark
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

models = {
    'SVM_RBF': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', SVC(C=10, gamma='scale', kernel='rbf', probability=True, random_state=RANDOM_STATE)),
    ]),
    'RandomForest': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(n_estimators=500, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    'ExtraTrees': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', ExtraTreesClassifier(n_estimators=500, random_state=RANDOM_STATE, class_weight='balanced')),
    ]),
    'GradientBoosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', GradientBoostingClassifier(random_state=RANDOM_STATE)),
    ]),
}

if XGBClassifier is not None:
    models['XGBoost'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.90,
            colsample_bytree=0.90,
            eval_metric='logloss',
            random_state=RANDOM_STATE,
        )),
    ])

if CatBoostClassifier is not None:
    models['CatBoost'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.05,
            verbose=False,
            random_seed=RANDOM_STATE,
        )),
    ])

benchmark_rows = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_pred = model.predict(X_test)
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred

    try:
        auc = roc_auc_score(y_test, y_prob)
    except ValueError:
        auc = np.nan

    benchmark_rows.append({
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'roc_auc': auc,
    })

benchmark_df = pd.DataFrame(benchmark_rows).sort_values(
    by=['roc_auc', 'f1_score', 'accuracy'],
    ascending=False,
).reset_index(drop=True)

benchmark_file = TABLE_DIR / '05_06_reca_model_benchmark.csv'
benchmark_df.to_csv(benchmark_file, index=False)

best_model_name = benchmark_df.loc[0, 'model']
best_model = fitted_models[best_model_name]

print('Benchmark saved to:', benchmark_file)
print('Best model:', best_model_name)
benchmark_df


In [ ]:
# ============================================================
# 6. Evaluation figures for the best model
# ============================================================

y_test_pred = best_model.predict(X_test)
y_test_prob = best_model.predict_proba(X_test)[:, 1]

print('Classification report for best model:')
print(classification_report(y_test, y_test_pred, target_names=['inactive_like', 'active_like']))

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title(f'Confusion Matrix: {best_model_name}')
plt.xticks([0, 1], ['inactive', 'active'])
plt.yticks([0, 1], ['inactive', 'active'])
plt.xlabel('Predicted label')
plt.ylabel('True label')
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha='center', va='center')
confusion_file = FIGURE_DIR / '05_06_best_model_confusion_matrix.png'
save_current_figure(confusion_file)
plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_test_prob)
roc_auc = roc_auc_score(y_test, y_test_prob)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.3f}')
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title(f'ROC Curve: {best_model_name}')
plt.legend(loc='lower right')
roc_file = FIGURE_DIR / '05_06_best_model_roc_curve.png'
save_current_figure(roc_file)
plt.show()


## 6. Load FDA fingerprint matrix and generate RecA active-like predictions

This section aligns FDA-approved drug fingerprints with the selected RecA training features. Missing fingerprint columns are filled with zero to avoid column mismatch errors.


In [ ]:
# ============================================================
# 7. Load FDA fingerprints and align features
# ============================================================

FDA_FINGERPRINT_FILE = first_existing_path(FDA_FINGERPRINT_CANDIDATES, 'FDA fingerprint file')
fda_fp_df = pd.read_csv(FDA_FINGERPRINT_FILE)

if 'Name' not in fda_fp_df.columns:
    raise ValueError("FDA fingerprint matrix must contain a 'Name' column from PaDEL.")

missing_features = [feature for feature in selected_features if feature not in fda_fp_df.columns]
if missing_features:
    print(f'Adding {len(missing_features)} missing FDA feature columns as zero-filled columns.')
    for feature in missing_features:
        fda_fp_df[feature] = 0

fda_X = clean_feature_matrix(fda_fp_df, selected_features)
print('FDA fingerprint matrix:', fda_fp_df.shape)
print('Aligned FDA matrix:', fda_X.shape)
fda_fp_df[['Name']].head()


In [ ]:
# ============================================================
# 8. Predict FDA active-like probability against RecA
# ============================================================

fda_prob = best_model.predict_proba(fda_X)[:, 1]
fda_pred_label = np.where(fda_prob >= ACTIVE_PROBABILITY_THRESHOLD, 'active_like', 'inactive_like')

fda_prediction_df = fda_fp_df.copy()
fda_prediction_df['predicted_probability_active'] = fda_prob
fda_prediction_df['predicted_label'] = fda_pred_label

# Create CID column if the PaDEL Name uses CIDxxxx format.
fda_prediction_df['cid'] = (
    fda_prediction_df['Name']
    .astype(str)
    .str.replace('CID', '', regex=False)
)
fda_prediction_df['cid'] = pd.to_numeric(fda_prediction_df['cid'], errors='coerce')

fda_prediction_df = fda_prediction_df.sort_values(
    'predicted_probability_active',
    ascending=False,
).reset_index(drop=True)

prediction_file = TABLE_DIR / '05_06_reca_fda_active_like_predictions.csv'
fda_prediction_df.to_csv(prediction_file, index=False)

print('FDA prediction table saved to:', prediction_file)
fda_prediction_df[['Name', 'cid', 'predicted_probability_active', 'predicted_label']].head(TOP_N_CANDIDATES)


In [ ]:
# ============================================================
# 9. Top-candidate probability plot
# ============================================================

top_candidates = fda_prediction_df.head(TOP_N_CANDIDATES).copy()
top_candidates['candidate_label'] = top_candidates['Name'].astype(str)

plt.figure(figsize=(8, 6))
plt.barh(
    top_candidates['candidate_label'][::-1],
    top_candidates['predicted_probability_active'][::-1],
)
plt.xlabel('Predicted RecA active-like probability')
plt.ylabel('FDA-approved candidate')
plt.title('Top FDA-Approved Drug Repurposing Candidates for RecA')
plt.xlim(0, 1)
probability_plot_file = FIGURE_DIR / '05_06_top_fda_reca_candidate_probabilities.png'
save_current_figure(probability_plot_file)
plt.show()


## 7. ADMET-likeness triage

This is a lightweight computational triage step. It does not replace full ADMET prediction software or experimental validation. It summarizes common drug-likeness ranges using molecular-weight, XLogP, TPSA, hydrogen-bond donor/acceptor, and rotatable-bond information when those columns are available.


In [ ]:
# ============================================================
# 10. ADMET-likeness scoring
# ============================================================

def find_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    lowered = {col.lower(): col for col in df.columns}
    for candidate in candidates:
        if candidate.lower() in lowered:
            return lowered[candidate.lower()]
    return None


def add_admet_likeness_scores(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    col_mw = find_column(result, ['molecular_weight', 'MolecularWeight', 'MW', 'full_mwt'])
    col_logp = find_column(result, ['xlogp', 'XLogP', 'alogp', 'LogP'])
    col_tpsa = find_column(result, ['tpsa', 'TPSA', 'psa'])
    col_hbd = find_column(result, ['hbd', 'HBondDonorCount', 'h_bond_donor_count'])
    col_hba = find_column(result, ['hba', 'HBondAcceptorCount', 'h_bond_acceptor_count'])
    col_rtb = find_column(result, ['rtb', 'RotatableBondCount', 'rotatable_bond_count'])

    available = {
        'MW': col_mw,
        'LogP': col_logp,
        'TPSA': col_tpsa,
        'HBD': col_hbd,
        'HBA': col_hba,
        'RTB': col_rtb,
    }
    print('Available ADMET columns:', {k: v for k, v in available.items() if v is not None})

    score = pd.Series(0, index=result.index, dtype=float)
    checks = pd.Series(0, index=result.index, dtype=float)

    def add_check(column: Optional[str], condition):
        nonlocal score, checks
        if column is None:
            return
        values = pd.to_numeric(result[column], errors='coerce')
        valid = values.notna()
        score.loc[valid] += condition(values.loc[valid]).astype(float)
        checks.loc[valid] += 1

    add_check(col_mw, lambda s: s <= 500)
    add_check(col_logp, lambda s: s <= 5)
    add_check(col_hbd, lambda s: s <= 5)
    add_check(col_hba, lambda s: s <= 10)
    add_check(col_tpsa, lambda s: s <= 140)
    add_check(col_rtb, lambda s: s <= 10)

    result['admet_rule_checks_available'] = checks.astype(int)
    result['admet_likeness_score'] = np.where(checks > 0, score / checks, np.nan)

    # Combined ranking: ML probability is primary; ADMET is supportive.
    result['combined_ml_admet_score'] = (
        0.75 * result['predicted_probability_active']
        + 0.25 * result['admet_likeness_score'].fillna(result['admet_likeness_score'].median())
    )

    return result.sort_values('combined_ml_admet_score', ascending=False).reset_index(drop=True)

admet_ranked_df = add_admet_likeness_scores(fda_prediction_df)
admet_file = TABLE_DIR / '05_06_reca_fda_predictions_with_admet_likeness.csv'
admet_ranked_df.to_csv(admet_file, index=False)

print('ADMET-ranked table saved to:', admet_file)
admet_ranked_df[['Name', 'cid', 'predicted_probability_active', 'admet_likeness_score', 'combined_ml_admet_score', 'predicted_label']].head(TOP_N_CANDIDATES)


In [ ]:
# ============================================================
# 11. Combined ML + ADMET score plot
# ============================================================

plot_df = admet_ranked_df.head(TOP_N_CANDIDATES).copy()
plt.figure(figsize=(8, 6))
plt.barh(plot_df['Name'].astype(str)[::-1], plot_df['combined_ml_admet_score'][::-1])
plt.xlabel('Combined ML + ADMET priority score')
plt.ylabel('FDA-approved candidate')
plt.title('RecA FDA Candidates Ranked by ML Prediction and ADMET-Likeness')
plt.xlim(0, 1)
admet_plot_file = FIGURE_DIR / '05_06_combined_ml_admet_candidate_ranking.png'
save_current_figure(admet_plot_file)
plt.show()


## 8. SHAP global interpretability

SHAP analysis is used to identify which molecular fingerprint descriptors most strongly influence the selected model. For tree-based models, `TreeExplainer` is preferred. For non-tree models such as SVM, the notebook uses a model-agnostic explainer on a small background subset to keep computation manageable.


In [ ]:
# ============================================================
# 12. SHAP helper functions
# ============================================================

def get_final_estimator(model):
    if isinstance(model, Pipeline):
        return model.named_steps.get('model')
    return model


def get_transformed_features(model: Pipeline, data: pd.DataFrame) -> np.ndarray:
    """Apply preprocessing steps before the final estimator."""
    if not isinstance(model, Pipeline):
        return data.values
    transformed = data.copy()
    for step_name, step in model.steps[:-1]:
        transformed = step.transform(transformed)
    return transformed


def compute_shap_values_for_best_model(model: Pipeline, X_reference: pd.DataFrame, X_explain: pd.DataFrame):
    if shap is None:
        raise ImportError('SHAP is not installed. Run: !pip install shap')

    estimator = get_final_estimator(model)
    X_ref_transformed = get_transformed_features(model, X_reference)
    X_exp_transformed = get_transformed_features(model, X_explain)

    tree_estimators = (
        RandomForestClassifier,
        ExtraTreesClassifier,
        GradientBoostingClassifier,
    )
    if XGBClassifier is not None:
        tree_estimators = tree_estimators + (XGBClassifier,)
    if CatBoostClassifier is not None:
        tree_estimators = tree_estimators + (CatBoostClassifier,)

    if isinstance(estimator, tree_estimators):
        explainer = shap.TreeExplainer(estimator)
        shap_values = explainer.shap_values(X_exp_transformed)
    else:
        # Model-agnostic fallback for SVM and other estimators.
        background = shap.sample(pd.DataFrame(X_ref_transformed, columns=selected_features), min(50, len(X_ref_transformed)), random_state=RANDOM_STATE)
        explain_frame = pd.DataFrame(X_exp_transformed, columns=selected_features)
        explainer = shap.Explainer(lambda z: estimator.predict_proba(z)[:, 1], background)
        shap_values = explainer(explain_frame)

    # Convert class-specific SHAP output into a 2D matrix for active-like class.
    if isinstance(shap_values, list):
        shap_matrix = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    elif hasattr(shap_values, 'values'):
        values = shap_values.values
        if values.ndim == 3:
            shap_matrix = values[:, :, 1]
        else:
            shap_matrix = values
    else:
        shap_matrix = shap_values

    return np.asarray(shap_matrix), X_exp_transformed


In [ ]:
# ============================================================
# 13. Compute SHAP values on the test set
# ============================================================

if shap is None:
    print('SHAP is not installed. Install shap to run this section.')
else:
    shap_matrix, X_test_transformed = compute_shap_values_for_best_model(best_model, X_train, X_test)
    shap_feature_df = pd.DataFrame(X_test_transformed, columns=selected_features)

    mean_abs_shap = pd.DataFrame({
        'feature': selected_features,
        'mean_abs_shap': np.abs(shap_matrix).mean(axis=0),
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    shap_importance_file = TABLE_DIR / '05_06_reca_global_shap_feature_importance.csv'
    mean_abs_shap.to_csv(shap_importance_file, index=False)

    print('SHAP importance table saved to:', shap_importance_file)
    display(mean_abs_shap.head(20))


In [ ]:
# ============================================================
# 14. SHAP global summary plots
# ============================================================

if shap is not None:
    plt.figure(figsize=(8, 6))
    top20 = mean_abs_shap.head(20).iloc[::-1]
    plt.barh(top20['feature'], top20['mean_abs_shap'])
    plt.xlabel('Mean |SHAP value|')
    plt.ylabel('Molecular fingerprint feature')
    plt.title(f'Global SHAP Feature Importance: {best_model_name}')
    shap_bar_file = FIGURE_DIR / '05_06_global_shap_feature_importance_bar.png'
    save_current_figure(shap_bar_file)
    plt.show()

    shap.summary_plot(
        shap_matrix,
        shap_feature_df,
        feature_names=selected_features,
        show=False,
        max_display=20,
    )
    shap_summary_file = FIGURE_DIR / '05_06_shap_summary_beeswarm.png'
    save_current_figure(shap_summary_file)
    plt.show()


## 9. Local SHAP explanation for top FDA candidates

This section explains why the model assigns high active-like probability to top FDA-approved candidates. The local SHAP tables are suitable for supplementary material.


In [ ]:
# ============================================================
# 15. Local SHAP explanation for top FDA candidates
# ============================================================

if shap is None:
    print('SHAP is not installed. Install shap to run this section.')
else:
    top_local = admet_ranked_df.head(TOP_N_SHAP_LOCAL).copy()
    top_names = top_local['Name'].astype(str).tolist()

    fda_explain_raw = fda_fp_df.set_index('Name').loc[top_names, selected_features].reset_index(drop=True)
    fda_explain_raw = clean_feature_matrix(fda_explain_raw, selected_features)

    fda_shap_matrix, fda_transformed = compute_shap_values_for_best_model(best_model, X_train, fda_explain_raw)
    fda_transformed_df = pd.DataFrame(fda_transformed, columns=selected_features)

    local_rows = []
    for candidate_index, candidate_name in enumerate(top_names):
        candidate_shap = pd.DataFrame({
            'candidate_name': candidate_name,
            'feature': selected_features,
            'feature_value': fda_transformed_df.iloc[candidate_index].values,
            'shap_value': fda_shap_matrix[candidate_index],
            'abs_shap_value': np.abs(fda_shap_matrix[candidate_index]),
        }).sort_values('abs_shap_value', ascending=False)

        local_rows.append(candidate_shap.head(15))

        # Waterfall-like horizontal contribution plot.
        plot_local = candidate_shap.head(10).iloc[::-1]
        plt.figure(figsize=(7, 5))
        plt.barh(plot_local['feature'], plot_local['shap_value'])
        plt.xlabel('SHAP contribution toward active-like probability')
        plt.ylabel('Feature')
        plt.title(f'Local SHAP Explanation: {candidate_name}')
        local_plot_file = FIGURE_DIR / f'05_06_local_shap_{candidate_name}.png'.replace('/', '_')
        save_current_figure(local_plot_file)
        plt.show()

    local_shap_df = pd.concat(local_rows, ignore_index=True)
    local_shap_file = TABLE_DIR / '05_06_local_shap_top_fda_candidates.csv'
    local_shap_df.to_csv(local_shap_file, index=False)

    print('Local SHAP table saved to:', local_shap_file)
    display(local_shap_df.head(30))


## 10. Final publication tables

The final table combines predicted active-like probability, ADMET-likeness score, and ranking information. This table can be used as the main candidate-prioritization output before docking or molecular dynamics validation.


In [ ]:
# ============================================================
# 16. Export final publication-ready candidate table
# ============================================================

final_columns_priority = [
    'Name',
    'cid',
    'predicted_probability_active',
    'predicted_label',
    'admet_rule_checks_available',
    'admet_likeness_score',
    'combined_ml_admet_score',
]

available_final_columns = [col for col in final_columns_priority if col in admet_ranked_df.columns]
other_metadata = [
    col for col in admet_ranked_df.columns
    if col not in available_final_columns and col not in selected_features
]

final_publication_table = admet_ranked_df[available_final_columns + other_metadata].copy()
final_publication_table.insert(0, 'final_rank', np.arange(1, len(final_publication_table) + 1))

final_table_file = TABLE_DIR / '05_06_final_reca_fda_shap_admet_publication_table.csv'
final_publication_table.to_csv(final_table_file, index=False)

print('Final publication table saved to:', final_table_file)
final_publication_table.head(TOP_N_CANDIDATES)


## 11. Manuscript-ready method description

Suggested wording for the Methods/Supplementary Methods section:

> FDA-approved compounds were represented using PaDEL molecular fingerprints and aligned to the RecA feature space obtained after low-variance filtering and F-score-based feature ranking. Multiple classical machine-learning classifiers were benchmarked using stratified train–test splitting, and the best-performing model was selected based on ROC-AUC and F1-score. FDA-approved drugs were then prioritized by predicted active-like probability against RecA. To improve interpretability, SHAP analysis was performed to identify globally important fingerprint descriptors and candidate-specific feature contributions. A lightweight ADMET-likeness triage based on common physicochemical rules was added as a secondary prioritization layer. The resulting candidates were ranked using a combined ML–ADMET score for subsequent docking and molecular simulation validation.


In [ ]:
# ============================================================
# 17. Output checklist
# ============================================================

output_checklist = {
    'model_benchmark': str(TABLE_DIR / '05_06_reca_model_benchmark.csv'),
    'prediction_table': str(TABLE_DIR / '05_06_reca_fda_active_like_predictions.csv'),
    'admet_ranked_table': str(TABLE_DIR / '05_06_reca_fda_predictions_with_admet_likeness.csv'),
    'final_publication_table': str(TABLE_DIR / '05_06_final_reca_fda_shap_admet_publication_table.csv'),
    'confusion_matrix': str(FIGURE_DIR / '05_06_best_model_confusion_matrix.png'),
    'roc_curve': str(FIGURE_DIR / '05_06_best_model_roc_curve.png'),
    'top_candidate_probability_plot': str(FIGURE_DIR / '05_06_top_fda_reca_candidate_probabilities.png'),
    'combined_ml_admet_plot': str(FIGURE_DIR / '05_06_combined_ml_admet_candidate_ranking.png'),
    'global_shap_bar_plot': str(FIGURE_DIR / '05_06_global_shap_feature_importance_bar.png'),
    'global_shap_beeswarm': str(FIGURE_DIR / '05_06_shap_summary_beeswarm.png'),
    'local_shap_table': str(TABLE_DIR / '05_06_local_shap_top_fda_candidates.csv'),
}

checklist_file = OUTPUT_DIR / '05_06_output_checklist.json'
with open(checklist_file, 'w', encoding='utf-8') as f:
    json.dump(output_checklist, f, indent=2)

print('Output checklist saved to:', checklist_file)
for key, value in output_checklist.items():
    print(f'- {key}: {value}')


# 7. Bayesian Good/Bad Fingerprint Profiling

This section follows the senior-style Bayesian fingerprint interpretation method. Each binary PaDEL fingerprint is classified as:

- **Good fingerprint**: enriched in active-like RecA inhibitors.
- **Bad fingerprint**: enriched in inactive-like compounds.
- **Neutral fingerprint**: no meaningful enrichment direction.

The method uses Laplace-smoothed conditional probabilities and log-likelihood ratios:

\[
LLR_j = \log \left( \frac{P(bit_j=1 \mid active)}{P(bit_j=1 \mid inactive)} \right)
\]

Positive values indicate active-enriched bits, while negative values indicate inactive-enriched bits.


In [ ]:
# ============================================================
# 18. Bayesian Good/Bad Fingerprint configuration
# ============================================================

BAYESIAN_DIR = OUTPUT_DIR / 'bayesian_good_bad_fingerprints'
BAYESIAN_TABLE_DIR = BAYESIAN_DIR / 'tables'
BAYESIAN_FIGURE_DIR = BAYESIAN_DIR / 'figures'

for folder in [BAYESIAN_DIR, BAYESIAN_TABLE_DIR, BAYESIAN_FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BAYESIAN_ALPHA = 1.0  # Laplace smoothing parameter
TOP_N_BAYESIAN_BITS = 20
BAYESIAN_WEIGHT_IN_FINAL_SCORE = 0.30

print('Bayesian output directory:', BAYESIAN_DIR)
print('Laplace smoothing alpha:', BAYESIAN_ALPHA)


In [ ]:
# ============================================================
# 19. Bayesian Good/Bad Fingerprint helper functions
# ============================================================

def get_binary_fingerprint_columns(
    df: pd.DataFrame,
    candidate_features: list[str],
    min_variance: float = 0.0,
) -> list[str]:
    """Return selected features that behave as binary PaDEL fingerprints."""
    binary_cols = []
    for col in candidate_features:
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col], errors='coerce').dropna()
        if values.empty:
            continue
        unique_values = set(values.unique().astype(float))
        if unique_values.issubset({0.0, 1.0}) and values.var() > min_variance:
            binary_cols.append(col)
    return binary_cols


def to_binary_fingerprint_matrix(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    """Convert selected fingerprint columns into a strict 0/1 matrix."""
    matrix = clean_feature_matrix(df, features)
    return (matrix > 0).astype(int)


def compute_bayesian_good_bad_table(
    X_binary: pd.DataFrame,
    y_binary: pd.Series,
    alpha: float = 1.0,
) -> pd.DataFrame:
    """Compute senior-style Bayesian Good/Bad Fingerprint statistics."""
    X_binary = X_binary.reset_index(drop=True).apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
    y_binary = pd.Series(y_binary).reset_index(drop=True).astype(int)

    active_mask = y_binary == 1
    inactive_mask = y_binary == 0

    n_active = int(active_mask.sum())
    n_inactive = int(inactive_mask.sum())

    if n_active == 0 or n_inactive == 0:
        raise ValueError(
            'Bayesian Good/Bad Fingerprint profiling requires both active-like and inactive-like classes.'
        )

    rows = []
    for feature in X_binary.columns:
        active_count = int(X_binary.loc[active_mask, feature].sum())
        inactive_count = int(X_binary.loc[inactive_mask, feature].sum())

        p_active = (active_count + alpha) / (n_active + 2 * alpha)
        p_inactive = (inactive_count + alpha) / (n_inactive + 2 * alpha)

        log_likelihood_ratio = float(np.log(p_active / p_inactive))
        enrichment_ratio = float(p_active / p_inactive)

        if log_likelihood_ratio > 0:
            fingerprint_class = 'good_active_enriched'
        elif log_likelihood_ratio < 0:
            fingerprint_class = 'bad_inactive_enriched'
        else:
            fingerprint_class = 'neutral'

        rows.append({
            'fingerprint': feature,
            'active_count': active_count,
            'inactive_count': inactive_count,
            'n_active': n_active,
            'n_inactive': n_inactive,
            'p_bit_given_active_smoothed': p_active,
            'p_bit_given_inactive_smoothed': p_inactive,
            'enrichment_ratio_active_vs_inactive': enrichment_ratio,
            'log_likelihood_ratio': log_likelihood_ratio,
            'abs_log_likelihood_ratio': abs(log_likelihood_ratio),
            'dataset_prevalence': float(X_binary[feature].mean()),
            'bayesian_fingerprint_class': fingerprint_class,
        })

    return (
        pd.DataFrame(rows)
        .sort_values(['abs_log_likelihood_ratio', 'dataset_prevalence'], ascending=[False, False])
        .reset_index(drop=True)
    )


def sigmoid(x):
    """Numerically stable sigmoid for Bayesian log scores."""
    x = np.asarray(x, dtype=float)
    return np.where(
        x >= 0,
        1 / (1 + np.exp(-x)),
        np.exp(x) / (1 + np.exp(x)),
    )


def compute_bayesian_sample_scores(
    X_binary: pd.DataFrame,
    bayesian_table: pd.DataFrame,
    class_prior_active: float,
) -> pd.DataFrame:
    """Score samples by summing present good/bad fingerprint log-likelihood contributions."""
    X_binary = X_binary.copy().apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

    weight_map = bayesian_table.set_index('fingerprint')['log_likelihood_ratio'].to_dict()
    common_features = [f for f in X_binary.columns if f in weight_map]
    if len(common_features) == 0:
        raise ValueError('No overlapping Bayesian fingerprint features were found for scoring.')

    weights = pd.Series({f: weight_map[f] for f in common_features})
    prior = np.clip(class_prior_active, 1e-6, 1 - 1e-6)
    bayesian_log_prior = np.log(prior / (1 - prior))

    raw_score = X_binary[common_features].dot(weights) + bayesian_log_prior
    probability = sigmoid(raw_score)

    good_features = bayesian_table.loc[
        bayesian_table['log_likelihood_ratio'] > 0, 'fingerprint'
    ].tolist()
    bad_features = bayesian_table.loc[
        bayesian_table['log_likelihood_ratio'] < 0, 'fingerprint'
    ].tolist()

    good_common = [f for f in good_features if f in X_binary.columns]
    bad_common = [f for f in bad_features if f in X_binary.columns]

    return pd.DataFrame({
        'bayesian_log_score': raw_score,
        'bayesian_probability_active_like': probability,
        'bayesian_good_fingerprint_count': X_binary[good_common].sum(axis=1) if good_common else 0,
        'bayesian_bad_fingerprint_count': X_binary[bad_common].sum(axis=1) if bad_common else 0,
    })


def save_bayesian_barplot(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    xlabel: str,
    output_file: Path,
):
    """Save a publication-style horizontal bar plot."""
    if df.empty:
        print(f'No data available for plot: {title}')
        return

    plot_df = df.copy().iloc[::-1]
    plt.figure(figsize=(8, 6))
    plt.barh(plot_df[y_col].astype(str), plot_df[x_col])
    plt.xlabel(xlabel)
    plt.ylabel('Fingerprint bit')
    plt.title(title)
    plt.tight_layout()
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print('Figure saved to:', output_file)
    plt.show()


In [ ]:
# ============================================================
# 20. Prepare binary RecA fingerprint matrix
# ============================================================

binary_fingerprint_features = get_binary_fingerprint_columns(training_df, selected_features)

if len(binary_fingerprint_features) == 0:
    print('No strict binary selected features detected. Selected features will be binarized using value > 0.')
    binary_fingerprint_features = [feature for feature in selected_features if feature in training_df.columns]

X_binary_all = to_binary_fingerprint_matrix(training_df, binary_fingerprint_features)

print('Binary fingerprint matrix shape:', X_binary_all.shape)
print('Number of Bayesian fingerprint features:', len(binary_fingerprint_features))
X_binary_all.head()


In [ ]:
# ============================================================
# 21. Fit Bayesian profile on training split only and evaluate hold-out set
# ============================================================

# Important publication safeguard:
# Bayesian statistics for evaluation are fitted using only X_train/y_train to avoid data leakage.
X_train_binary = X_binary_all.loc[X_train.index, binary_fingerprint_features].copy()
X_test_binary = X_binary_all.loc[X_test.index, binary_fingerprint_features].copy()

bayesian_train_table = compute_bayesian_good_bad_table(
    X_binary=X_train_binary,
    y_binary=y_train,
    alpha=BAYESIAN_ALPHA,
)

class_prior_active_train = float(y_train.mean())

train_bayesian_scores = compute_bayesian_sample_scores(
    X_train_binary,
    bayesian_train_table,
    class_prior_active_train,
)
test_bayesian_scores = compute_bayesian_sample_scores(
    X_test_binary,
    bayesian_train_table,
    class_prior_active_train,
)

train_pred = (
    train_bayesian_scores['bayesian_probability_active_like'] >= ACTIVE_PROBABILITY_THRESHOLD
).astype(int)
test_pred = (
    test_bayesian_scores['bayesian_probability_active_like'] >= ACTIVE_PROBABILITY_THRESHOLD
).astype(int)

bayesian_eval = pd.DataFrame([
    {
        'dataset': 'training',
        'accuracy': accuracy_score(y_train, train_pred),
        'precision': precision_score(y_train, train_pred, zero_division=0),
        'recall': recall_score(y_train, train_pred, zero_division=0),
        'f1_score': f1_score(y_train, train_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_train, train_bayesian_scores['bayesian_probability_active_like'])
        if len(np.unique(y_train)) > 1 else np.nan,
    },
    {
        'dataset': 'holdout_test',
        'accuracy': accuracy_score(y_test, test_pred),
        'precision': precision_score(y_test, test_pred, zero_division=0),
        'recall': recall_score(y_test, test_pred, zero_division=0),
        'f1_score': f1_score(y_test, test_pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, test_bayesian_scores['bayesian_probability_active_like'])
        if len(np.unique(y_test)) > 1 else np.nan,
    },
])

bayesian_train_table_file = BAYESIAN_TABLE_DIR / '07_bayesian_train_split_good_bad_fingerprint_scores.csv'
bayesian_eval_file = BAYESIAN_TABLE_DIR / '07_bayesian_holdout_evaluation.csv'

bayesian_train_table.to_csv(bayesian_train_table_file, index=False)
bayesian_eval.to_csv(bayesian_eval_file, index=False)

print('Training-split Bayesian fingerprint table saved to:', bayesian_train_table_file)
print('Bayesian hold-out evaluation saved to:', bayesian_eval_file)
display(bayesian_eval)

print('Hold-out classification report for Bayesian Good/Bad Fingerprint classifier:')
print(classification_report(y_test, test_pred, target_names=['inactive_like', 'active_like'], zero_division=0))


In [ ]:
# ============================================================
# 22. Fit final Bayesian Good/Bad profile using all labeled RecA data
# ============================================================

# After the unbiased hold-out evaluation above, a final full-data Bayesian profile
# is fitted for interpretation and FDA candidate scoring.
bayesian_table = compute_bayesian_good_bad_table(
    X_binary=X_binary_all,
    y_binary=y,
    alpha=BAYESIAN_ALPHA,
)

bayesian_good_df = bayesian_table[
    bayesian_table['bayesian_fingerprint_class'] == 'good_active_enriched'
].copy()
bayesian_bad_df = bayesian_table[
    bayesian_table['bayesian_fingerprint_class'] == 'bad_inactive_enriched'
].copy()

bayesian_all_file = BAYESIAN_TABLE_DIR / '07_final_bayesian_all_fingerprint_scores.csv'
bayesian_good_file = BAYESIAN_TABLE_DIR / '07_final_bayesian_good_active_enriched_fingerprints.csv'
bayesian_bad_file = BAYESIAN_TABLE_DIR / '07_final_bayesian_bad_inactive_enriched_fingerprints.csv'
summary_file = BAYESIAN_TABLE_DIR / '07_final_bayesian_good_bad_summary_table.csv'

bayesian_table.to_csv(bayesian_all_file, index=False)
bayesian_good_df.to_csv(bayesian_good_file, index=False)
bayesian_bad_df.to_csv(bayesian_bad_file, index=False)

summary_df = pd.DataFrame({
    'metric': [
        'total_binary_fingerprints',
        'good_active_enriched_fingerprints',
        'bad_inactive_enriched_fingerprints',
        'neutral_fingerprints',
        'laplace_smoothing_alpha',
        'active_prior_full_dataset',
    ],
    'value': [
        len(binary_fingerprint_features),
        len(bayesian_good_df),
        len(bayesian_bad_df),
        int((bayesian_table['bayesian_fingerprint_class'] == 'neutral').sum()),
        BAYESIAN_ALPHA,
        float(y.mean()),
    ],
})
summary_df.to_csv(summary_file, index=False)

print('Final Bayesian fingerprint score table saved to:', bayesian_all_file)
print('Good fingerprint table saved to:', bayesian_good_file)
print('Bad fingerprint table saved to:', bayesian_bad_file)
print('Summary table saved to:', summary_file)

display(summary_df)
display(bayesian_table.head(20))


In [ ]:
# ============================================================
# 23. Visualize top Bayesian good and bad fingerprints
# ============================================================

top_good = (
    bayesian_good_df
    .sort_values('log_likelihood_ratio', ascending=False)
    .head(TOP_N_BAYESIAN_BITS)
)
top_bad = (
    bayesian_bad_df
    .sort_values('log_likelihood_ratio', ascending=True)
    .head(TOP_N_BAYESIAN_BITS)
)

good_plot_file = BAYESIAN_FIGURE_DIR / '07_top20_good_active_enriched_bayesian_fingerprints.png'
bad_plot_file = BAYESIAN_FIGURE_DIR / '07_top20_bad_inactive_enriched_bayesian_fingerprints.png'

save_bayesian_barplot(
    top_good,
    x_col='log_likelihood_ratio',
    y_col='fingerprint',
    title='Top active-enriched Good Fingerprints for RecA',
    xlabel='Log-likelihood ratio: active vs inactive',
    output_file=good_plot_file,
)

save_bayesian_barplot(
    top_bad.assign(abs_bad_llr=top_bad['log_likelihood_ratio'].abs()),
    x_col='abs_bad_llr',
    y_col='fingerprint',
    title='Top inactive-enriched Bad Fingerprints for RecA',
    xlabel='Absolute log-likelihood ratio: inactive enrichment',
    output_file=bad_plot_file,
)


In [ ]:
# ============================================================
# 24. Apply Bayesian Good/Bad Fingerprint scoring to FDA candidates
# ============================================================

fda_binary_features = [feature for feature in binary_fingerprint_features if feature in fda_fp_df.columns]

if len(fda_binary_features) == 0:
    raise ValueError('None of the Bayesian fingerprint features were found in the FDA fingerprint table.')

fda_binary = to_binary_fingerprint_matrix(fda_fp_df, fda_binary_features)

class_prior_active_full = float(y.mean())
fda_bayesian_scores = compute_bayesian_sample_scores(
    X_binary=fda_binary,
    bayesian_table=bayesian_table[bayesian_table['fingerprint'].isin(fda_binary_features)],
    class_prior_active=class_prior_active_full,
)

# Use the ADMET-ranked table as the base if it exists; otherwise use the ML prediction table.
if 'admet_ranked_df' in globals():
    fda_bayesian_ranked = admet_ranked_df.copy().reset_index(drop=True)
else:
    fda_bayesian_ranked = fda_prediction_df.copy().reset_index(drop=True)

fda_bayesian_ranked = pd.concat(
    [fda_bayesian_ranked, fda_bayesian_scores.reset_index(drop=True)],
    axis=1,
)

if 'combined_ml_admet_score' in fda_bayesian_ranked.columns:
    base_score = pd.to_numeric(fda_bayesian_ranked['combined_ml_admet_score'], errors='coerce')
elif 'predicted_probability_active' in fda_bayesian_ranked.columns:
    base_score = pd.to_numeric(fda_bayesian_ranked['predicted_probability_active'], errors='coerce')
else:
    base_score = pd.Series(0.0, index=fda_bayesian_ranked.index)

base_score = base_score.fillna(base_score.median() if base_score.notna().any() else 0.0)

fda_bayesian_ranked['combined_ml_admet_bayesian_score'] = (
    (1 - BAYESIAN_WEIGHT_IN_FINAL_SCORE) * base_score
    + BAYESIAN_WEIGHT_IN_FINAL_SCORE * fda_bayesian_ranked['bayesian_probability_active_like']
)

fda_bayesian_ranked = (
    fda_bayesian_ranked
    .sort_values('combined_ml_admet_bayesian_score', ascending=False)
    .reset_index(drop=True)
)
fda_bayesian_ranked.insert(0, 'bayesian_integrated_rank', np.arange(1, len(fda_bayesian_ranked) + 1))

fda_bayesian_file = BAYESIAN_TABLE_DIR / '07_fda_candidates_ranked_with_bayesian_good_bad_fingerprints.csv'
fda_bayesian_ranked.to_csv(fda_bayesian_file, index=False)

print('FDA Bayesian-integrated candidate table saved to:', fda_bayesian_file)
display(fda_bayesian_ranked.head(TOP_N_CANDIDATES))


In [ ]:
# ============================================================
# 25. Plot Bayesian-integrated RecA FDA candidate ranking
# ============================================================

plot_bayes_fda = fda_bayesian_ranked.head(TOP_N_CANDIDATES).copy()
name_col = 'Name' if 'Name' in plot_bayes_fda.columns else plot_bayes_fda.columns[1]
plot_bayes_fda['candidate_label'] = plot_bayes_fda[name_col].astype(str)

bayesian_ranking_plot_file = BAYESIAN_FIGURE_DIR / '07_fda_candidates_ml_admet_bayesian_ranking.png'

plt.figure(figsize=(8, 7))
plt.barh(
    plot_bayes_fda['candidate_label'][::-1],
    plot_bayes_fda['combined_ml_admet_bayesian_score'][::-1],
)
plt.xlabel('Combined ML–ADMET–Bayesian score')
plt.ylabel('FDA candidate')
plt.title('Top FDA candidates prioritized by ML, ADMET, and Bayesian fingerprints')
plt.tight_layout()
plt.savefig(bayesian_ranking_plot_file, dpi=300, bbox_inches='tight')
print('Figure saved to:', bayesian_ranking_plot_file)
plt.show()


## 26. Manuscript-ready interpretation note

The Bayesian Good/Bad Fingerprint analysis provides an interpretable structural fingerprint layer complementary to the ML and SHAP analyses. Fingerprints enriched in the active-like RecA class were defined as **Good fingerprints**, whereas fingerprints enriched in the inactive-like class were defined as **Bad fingerprints**. Laplace smoothing was applied to avoid unstable probability estimates for sparse fingerprint bits. For publication integrity, the Bayesian classifier was evaluated on the same hold-out split used in the ML workflow using a training-split Bayesian profile. A final full-data Bayesian profile was then generated for interpretation and FDA candidate ranking.


In [ ]:
# ============================================================
# 27. Updated output checklist including Bayesian Good/Bad Fingerprints
# ============================================================

updated_output_checklist = dict(output_checklist) if 'output_checklist' in globals() else {}
updated_output_checklist.update({
    'bayesian_train_split_fingerprint_scores': str(BAYESIAN_TABLE_DIR / '07_bayesian_train_split_good_bad_fingerprint_scores.csv'),
    'bayesian_holdout_evaluation': str(BAYESIAN_TABLE_DIR / '07_bayesian_holdout_evaluation.csv'),
    'bayesian_final_all_fingerprint_scores': str(BAYESIAN_TABLE_DIR / '07_final_bayesian_all_fingerprint_scores.csv'),
    'bayesian_final_good_fingerprints': str(BAYESIAN_TABLE_DIR / '07_final_bayesian_good_active_enriched_fingerprints.csv'),
    'bayesian_final_bad_fingerprints': str(BAYESIAN_TABLE_DIR / '07_final_bayesian_bad_inactive_enriched_fingerprints.csv'),
    'bayesian_final_summary_table': str(BAYESIAN_TABLE_DIR / '07_final_bayesian_good_bad_summary_table.csv'),
    'fda_bayesian_integrated_ranking': str(BAYESIAN_TABLE_DIR / '07_fda_candidates_ranked_with_bayesian_good_bad_fingerprints.csv'),
    'top_good_bayesian_fingerprint_plot': str(BAYESIAN_FIGURE_DIR / '07_top20_good_active_enriched_bayesian_fingerprints.png'),
    'top_bad_bayesian_fingerprint_plot': str(BAYESIAN_FIGURE_DIR / '07_top20_bad_inactive_enriched_bayesian_fingerprints.png'),
    'fda_ml_admet_bayesian_ranking_plot': str(BAYESIAN_FIGURE_DIR / '07_fda_candidates_ml_admet_bayesian_ranking.png'),
})

updated_checklist_file = OUTPUT_DIR / '05_06_07_updated_output_checklist_with_bayesian.json'
with open(updated_checklist_file, 'w', encoding='utf-8') as file:
    json.dump(updated_output_checklist, file, indent=2)

print('Updated output checklist saved to:', updated_checklist_file)
for key, value in updated_output_checklist.items():
    print(f'- {key}: {value}')


## 28. Final quality-control checklist

Before submitting the notebook to a supervisor or including it as a publication supplement, confirm that:

1. All upstream files from notebooks 1–4 are available in the expected `outputs/` folders.
2. The notebook runs from top to bottom without manual variable injection.
3. All exported CSV tables and PNG figures are generated.
4. The hold-out Bayesian evaluation is reported separately from the full-data Bayesian interpretation table.
5. The final candidate ranking is described as **computational prioritization**, not experimental validation.
